In [ ]:

# implements the biographical profiling pipeline:
    # define the patient profile schema
    # build the query generation module (src/profiling.py)
    # build the full profiling-to-retrieval pipeline

import sys
import os
from dotenv import load_dotenv

# add project root to path
sys.path.insert(0, os.path.abspath(".."))
load_dotenv("../.env")

Define patient profile schema
patient profile is a dictionary with these fields:
- name: patient full name
- birth_year: year of birth (used to compute the reminiscence bump between ages 15–25)
- hometown: city and state (to target regional music)
- cultural_background: ethnic/cultural identity ( to target genre/artist community)
- life_events: list of {year, event} dicts mapped to music of those moments

In [ ]:
# sample patient profile used for testing in this notebook
sample_profile = {
    "name": "James Wilson",
    "birth_year": 1948,
    "hometown": "Detroit, Michigan",
    "cultural_background": "African American",
    "life_events": [
        {"year": 1966, "event": "Graduated from Cass Tech High School"},
        {"year": 1968, "event": "Drafted into the Vietnam War"},
        {"year": 1971, "event": "Married Dorothy in Detroit"},
        {"year": 1975, "event": "First child born"},
        {"year": 1980, "event": "Moved to Atlanta for work"},
    ]
}

# derived fields (computed automatically in profiling functions)
# bump = 15-25
bump_start = sample_profile["birth_year"] + 15 
bump_end   = sample_profile["birth_year"] + 25
print(f"Patient: {sample_profile['name']}")
print(f"Hometown: {sample_profile['hometown']}")
print(f"Cultural background: {sample_profile['cultural_background']}")
print(f"Reminiscence bump: {bump_start}–{bump_end} (ages 15–25)")
print(f"Life events:")
for event in sample_profile["life_events"]:
    print(f"  {event['year']}: {event['event']}")

Query generation module

generate_queries() takes a patient profile and uses GPT-4o to produce 5 to 8 targeted retrieval queries. Each query should surface songs that are:
- From the patient reminiscence bump years
- Relevant to  cultural background and geographic region
- Tied to specific life events

The function is in [src/profiling.py](../src/profiling.py)

In [ ]:
from src.profiling import generate_queries

# Test to generate retrieval queries for the sample profile
queries = generate_queries(sample_profile)

print(f"Generated {len(queries)} queries for {sample_profile['name']}:\n")
for i, q in enumerate(queries, 1):
    print(f"  {i}. {q}")

Full profiling -> retrieval pipeline

profile_to_context() connects everything together:
1. Calls generate_queries() to produce targeted queries
2. Runs query through the FAISS retrieval system
3. Combines and deduplicates results, keeping the top-scored unique songs

In [ ]:
from src.retrieval import load_retrieval_system
from src.profiling import profile_to_context

INDEX_PATH = "../data/index/songs.index"
KB_PATH    = "../data/processed/knowledge_base.csv"

# load FAISS index + knowledge base
index, df = load_retrieval_system(INDEX_PATH, KB_PATH)
print(f"loaded index with {index.ntotal} vectors and knowledge base with {len(df)} songs.")

In [ ]:
# run the full pipeline: profile → queries → retrieved songs
retrieved, queries_used = profile_to_context(sample_profile, index, df, k_per_query=10, total_k=20)

print(f"retrieved {len(retrieved)} unique songs after deduplication:\n")
print(retrieved[["song", "artist", "year", "similarity_score", "source_query"]].to_string(index=False))